# Session 01 — From Relative Logits to Predictive Surprise

**Deep question:** Why do relative scores, rather than absolute logits, determine a model's belief and the surprise assigned to an observed token?

We will use the cycle **predict → implement → perturb → interpret → bound the evidence**. Do not rush to run every cell. Stop at each prediction checkpoint and discuss your model of the mechanism first.

This notebook uses only Python's standard library.

## 0. The objects we are studying

At one token position:

- the model emits one **logit** for every vocabulary candidate: `logits.shape == [V]`;
- softmax converts those scores into a probability distribution: `probabilities.shape == [V]`;
- the **label** is one integer ID naming the observed next token: a scalar;
- negative log-likelihood reads the probability at that label.

Later, batching these objects produces logits `[B,T,V]` and labels `[B,T]` before causal alignment.

In [ ]:
from math import exp, isclose, log

TOKENS = ["cat", "dog", "slept"]
LOGITS = [2.0, 1.0, -1.0]
LABEL = 1  # the observed next token is 'dog'

list(zip(TOKENS, LOGITS)), TOKENS[LABEL]

## 1. Prediction checkpoint — normalized competition

Before coding, discuss these questions qualitatively:

1. Which token should receive the greatest probability, and why?
2. If we add `1000` to every logit, should the probabilities change?
3. If only the `dog` logit increases, which other probabilities must change?

Do not calculate the exact values yet. State the mechanism you expect.

## 2. Implement stable softmax

Softmax is `p_i = exp(z_i) / sum_j exp(z_j)`. Subtracting the maximum logit before exponentiation leaves the probabilities unchanged while preventing overflow. Replace only the `...` expressions.

In [ ]:
def stable_softmax(logits):
    maximum = ...                 # largest logit
    weights = ...                 # exp(z_i - maximum) for every z_i
    total = ...                   # sum of the positive weights
    probabilities = ...           # normalize every weight by total
    return probabilities


probabilities = stable_softmax(LOGITS)
list(zip(TOKENS, probabilities))

In [ ]:
assert len(probabilities) == len(TOKENS)
assert all(0.0 < p < 1.0 for p in probabilities)
assert isclose(sum(probabilities), 1.0, rel_tol=0.0, abs_tol=1e-12)
assert probabilities.index(max(probabilities)) == LOGITS.index(max(LOGITS))
print("The output is a normalized distribution and preserves the score ranking.")

## 3. Perturbation — absolute height versus relative gaps

We now compare the original logits with logits shifted upward by the same constant. This is not merely a numerical trick: it tests what information softmax preserves.

In [ ]:
shifted_logits = [z + 1000.0 for z in LOGITS]
shifted_probabilities = stable_softmax(shifted_logits)

for token, before, after in zip(TOKENS, probabilities, shifted_probabilities):
    print(f"{token:>6}: before={before:.8f} after={after:.8f}")

assert all(isclose(a, b, rel_tol=0.0, abs_tol=1e-12)
           for a, b in zip(probabilities, shifted_probabilities))

In [ ]:
def naive_softmax(logits):
    weights = [exp(z) for z in logits]
    return [weight / sum(weights) for weight in weights]

try:
    naive_softmax(shifted_logits)
except OverflowError as error:
    print("Naive implementation failed:", type(error).__name__)

print("Stable implementation still works:", stable_softmax(shifted_logits))

### Interpretation checkpoint

Explain why adding the same constant changes every raw score but changes no modeled belief. What does this reveal about interpreting one isolated logit?

## 4. Labels and negative log-likelihood

The label is not another model prediction. It is evidence from the dataset: the integer ID of the token that actually followed the context. Implement `-log(p_label)`.

In [ ]:
def nll_from_logits(logits, label):
    probabilities = ...           # call stable_softmax
    target_probability = ...      # index with the integer label
    loss = ...                    # negative natural logarithm
    return loss, target_probability


loss, target_probability = nll_from_logits(LOGITS, LABEL)
print("observed token:", TOKENS[LABEL])
print("target probability:", target_probability)
print("target surprise / NLL:", loss)

### Deep question

The highest-scoring token was `cat`, but the observed label is `dog`. Why does the loss need the complete competition among tokens rather than only the raw `dog` logit? Consider what would happen if every logit increased together.

## 5. Code the logit gradient `p - q`

For one-hot cross-entropy, the derivative with respect to logit `z_i` is `p_i - q_i`. The target coordinate has `q_i = 1`; every non-target coordinate has `q_i = 0`.

In [ ]:
def logit_gradient(logits, label):
    probabilities = ...
    target = ...                   # a one-hot list with the same length
    gradient = ...                 # elementwise p_i - q_i
    return probabilities, target, gradient


probabilities, target, gradient = logit_gradient(LOGITS, LABEL)
for token, p_i, q_i, grad_i in zip(TOKENS, probabilities, target, gradient):
    print(f"{token:>6}: p={p_i:.6f} q={q_i:.0f} dL/dz={grad_i:+.6f}")

assert isclose(sum(gradient), 0.0, rel_tol=0.0, abs_tol=1e-12)

### Interpretation checkpoint

Gradient descent subtracts the gradient. Explain why this raises the target logit and lowers every non-target logit. Then ask a harder question: why does one-hot supervision on one example not force the trained model to assign probability one to that token in every context?

## 6. From average surprise to perplexity

Perplexity exponentiates mean token NLL. It can be interpreted as an effective equal-choice branching factor under a fixed evaluation contract.

In [ ]:
def perplexity(token_losses):
    mean_nll = ...                 # arithmetic mean of valid token losses
    return ...                     # exp(mean_nll)


equal_four_way_loss = -log(0.25)
print(perplexity([equal_four_way_loss] * 5))
assert isclose(perplexity([equal_four_way_loss] * 5), 4.0, abs_tol=1e-12)

## 7. Evidence boundary — write before concluding

Complete these statements in your own words:

- **Observation:** After adding the same constant to every logit, ...
- **Mechanistic interpretation:** This happens because ...
- **Observation:** For one-hot cross-entropy, the target gradient was ... while non-target gradients were ...
- **What this notebook supports:** ...
- **What it does not prove about a trained language model:** ...

Bring the completed statements and any surprising output back to the guided discussion before Session 2.

## Final synthesis question

Softmax ignores a shared shift in all logits, yet cross-entropy can still strongly punish a confident error. How can both facts be true at the same time? Frame your answer in terms of **relative gaps**, **target probability**, and **gradient direction**.